<a href="https://colab.research.google.com/github/Innovatewithapple/CNNProjects/blob/main/VegetableImageDatasetPytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms,datasets
from torch.utils.data import DataLoader
import os
import warnings
from google.colab import userdata
import timm
import pandas as pd
import numpy as np
from PIL import Image,ImageFile

In [4]:
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

In [5]:
!kaggle datasets download -d misrakahmed/vegetable-image-dataset

Dataset URL: https://www.kaggle.com/datasets/misrakahmed/vegetable-image-dataset
License(s): CC-BY-SA-4.0
100% 534M/534M [00:25<00:00, 21.9MB/s]



In [6]:
!unzip -q vegetable-image-dataset.zip -d data_folder/

In [18]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pinmemory = (True if device == 'cuda' else False)
print(device)
print(pinmemory)

cpu
False


In [12]:
train_loc = '/content/data_folder/Vegetable Images/train'
val_loc = '/content/data_folder/Vegetable Images/validation'
test = '/content/data_folder/Vegetable Images/train/Bean'

In [14]:
sizes = []

# Scan the first 50 images for a quick check
for file in os.listdir(test)[:50]:
    if file.endswith(('.jpg', '.png', '.jpeg')):
        img = Image.open(os.path.join(test, file))
        sizes.append(img.size) # (width, height)

# Analyze the results
min_size = min(sizes)
max_size = max(sizes)
avg_size = (sum(w for w, h in sizes)//len(sizes), sum(h for w, h in sizes)//len(sizes))

print(f"📏 Smallest Image: {min_size}")
print(f"📏 Largest Image: {max_size}")
print(f"📏 Average Size: {avg_size}")

📏 Smallest Image: (224, 224)
📏 Largest Image: (224, 224)
📏 Average Size: (224, 224)


In [15]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])
val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [22]:
# 1. This is the secret: Turn the Warning into a Crash so 'except' catches it
warnings.filterwarnings("error", category=UserWarning)
count = 0
for root, dirs, files in os.walk(train_loc):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            file_path = os.path.join(root, file)
            try:
                img = Image.open(file_path)
                img.load()  # <--- This is the "Truth" test. It forces a full read.
            except Exception as e:
                print(f"❌ Deleting incomplete/corrupt image: {file_path}")
                os.remove(file_path)
                count += 1
# 2. IMPORTANT: Turn warnings back to normal after cleaning
warnings.resetwarnings()
print(f"🧹 Cleanup finished. Total deleted: {count}")

🧹 Cleanup finished. Total deleted: 0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [16]:
train_data = datasets.ImageFolder(root=train_loc,transform=train_transform)
val_data = datasets.ImageFolder(root=val_loc,transform=val_transform)

In [19]:
train_loader = DataLoader(dataset=train_data,batch_size=32,shuffle=True,pin_memory=pinmemory)
val_loader = DataLoader(dataset=val_data,batch_size=32,shuffle=False,pin_memory=pinmemory)

In [ ]:
model = timm.create_model(model_name='tf_efficientnetv2_s',pretrained=True,num_classes=0)

In [26]:
custom_head = nn.Sequential(
    nn.LazyLinear(256),
    nn.ReLU(),
    nn.Linear(256,15)
)

In [45]:
for name,children in model.named_children():
  print(name)

conv_stem
bn1
blocks
conv_head
bn2
global_pool
classifier


In [50]:
for param in model.parameters():
  param.requires_grad = False # freezed all the params of model

for subFolder in list(model.blocks)[-2:]:
  for param in subFolder.parameters():
    param.requires_grad = True

for param in custom_head.parameters():
  param.requires_grad = True

In [51]:
final_model = nn.Sequential(
    model,
    custom_head
).to(device)

In [52]:
optimizer = optim.Adam(params=final_model.parameters(),lr=0.001)
loss_fn = nn.CrossEntropyLoss()

In [61]:
tempimages,templabels = next(iter(train_loader))
print(templabels.shape)

torch.Size([32])


In [62]:
epochs = 10

for epoch in range(epochs):
  final_model.train()
  train_loss = 0
  train_correct = 0
  train_total = 0

  for images,labels in train_loader:
    images,labels = images.to(device), labels.to(device)

    optimizer.zero_grad()

    output = final_model(images)
    loss = loss_fn(output,labels)

    loss.backward()
    optimizer.step()

    train_loss += loss.item()
    pred = torch.argmax(output,dim=1) #here output gives [batch_size,classes] so we use classes
    train_correct += (pred == labels).sum().item()
    train_total += labels.size(0)

  train_accuracy = train_correct/train_total
  train_losses = train_loss / len(train_loader)

  final_model.eval()
  val_loss = 0
  val_correct = 0
  val_total = 0
  with torch.no_grad():
   for images,labels in val_loader:
    images,labels = images.to(device), labels.to(device)


    output = final_model(images)
    loss = loss_fn(output,labels)

    val_loss += loss.item()
    pred = torch.argmax(output,dim=1) #here output gives [batch_size,classes] so we use classes
    val_correct += (pred == labels).sum().item()
    val_total += labels.size(0)

   val_accuracy = val_correct/val_total
   val_losses = val_loss / len(val_loader)

   print(f'\nEpochs: {epoch+1}/{epochs}...')
   print(f'\ntrain_acc: {train_accuracy} | train_loss: {train_losses}')
   print(f'\nval_acc: {val_accuracy} | val_loss: {val_losses}')


KeyboardInterrupt: 